In [ ]:
import pandas as pd
import csv
import math
import operator
from nltk import word_tokenize

In [ ]:
test_questions =  pd.read_csv('../Data/cpp_goldset.csv').Question
test_questions

In [ ]:
corpus_questions_and_answers = pd.read_csv('../Data/cpp_goldset.csv')
corpus_questions_and_answers

In [ ]:
def create_question_corpus(filename):
    df = pd.read_csv("../Data/"+filename+"_corpus.csv")
    corpus = df[['Question']]
    corpus.to_csv("./question_corpus/"+filename+".txt", index=False)

In [ ]:
filenames = ["complete"]
for name in filenames:
    create_question_corpus(name)

# Build IDF Vocabulary

In [ ]:
def write_list_to_csv(list_tmp, csv_fpath, header):
    with open(csv_fpath, 'w', newline='', encoding="utf-8") as myfile:
        wr = csv.writer(myfile)
        wr.writerow(header)
        for x in list_tmp:
            try:
                if(isinstance(x, int)):
                    x=[x]
                wr.writerow(x)
            except Exception as e:
                print(("Error %s" % e))
    print(("Write %s successfully!" % csv_fpath))

In [ ]:
def build_idf_vocab(filename):
    df = pd.read_csv("./question_corpus/"+filename+".txt")
    corpus = df[['Question']]
    qlist = corpus.values.tolist()
    total_num = len(qlist)
    voc = {}
    count = 0
    for q in qlist:
        if q is not None :
            try :
                title_wlist = word_tokenize(str(q[0]))
                cur_word_set = set()
                for w in title_wlist:
                    if w not in cur_word_set:
                        cur_word_set.add(w)
                        if w not in list(voc.keys()):
                            voc[w] = 1.0
                        else:
                            voc[w] = voc[w] + 1.0
                count += 1
                if count % 10000 == 0:
                    print('processing %s unit...' % count, get_current_time())
            except Exception as e:
                print(("Error %s" % e))
           
    for key in voc.keys():
        idf = math.log(total_num / (voc[key] + 1.0))
        voc[key] = idf
    sorted_voc = sorted(list(voc.items()), key=operator.itemgetter(1))
    return sorted_voc

In [ ]:
filenames = ["java","python","js","cpp"]
for name in filenames:
    vocab = build_idf_vocab(name)
    fpath = './idf_vocab/'+name+'.csv'
    header = ['word', 'idf']
    write_list_to_csv(vocab, fpath, header)